# 02 - Fine-tuning Whisper pour l'ASR Darija

Ce notebook fine-tune **deux modeles Whisper-small separes** :
1. Darija -> script **Latin** (Arabizi)
2. Darija -> script **Arabe**

**A executer sur Google Colab avec GPU (Runtime > Modifier le type d'execution > GPU, idealement T4).**

Le notebook sauvegarde des checkpoints intermediaires sur Drive a intervalles reguliers,
ce qui permet de reprendre l'entrainement si la session Colab se deconnecte en cours de route.


## 1. Installation des dependances

In [ ]:
!pip install -q transformers accelerate evaluate jiwer datasets soundfile librosa

In [ ]:
import os
import torch
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_from_disk, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Connexion Drive + recuperation du projet (scripts/notebooks via GitHub)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = "/content/drive/MyDrive/darija-transcription-translation"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("PROJECT_DIR =", PROJECT_DIR)

In [ ]:
repo_url = "https://github.com/moprh/darija-transcription-translation.git"
clone_path = os.path.join(PROJECT_DIR, "repo")

if not os.path.exists(clone_path):
    !git clone {repo_url} {clone_path}
else:
    !cd {clone_path} && git pull

print("Projet disponible dans :", clone_path)

## 3. Configuration generale

`SCRIPT_TARGET` controle quel modele on entraine dans cette execution du notebook :
- `"latin"` -> fine-tune sur `data/processed/asr_latin`
- `"arabic"` -> fine-tune sur `data/processed/asr_arabic`

**Pour entrainer les deux modeles**, executer le notebook une fois avec `SCRIPT_TARGET = "latin"`,
puis a nouveau (Runtime > Redemarrer la session, ou juste re-executer depuis cette cellule)
avec `SCRIPT_TARGET = "arabic"`. Les checkpoints sont sauvegardes dans des dossiers separes
donc il n'y a pas de risque d'ecrasement.


In [ ]:
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
SCRIPT_TARGET = "latin"  # changer en "arabic" pour le second entrainement
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

assert SCRIPT_TARGET in ("latin", "arabic"), "SCRIPT_TARGET doit etre 'latin' ou 'arabic'"

WHISPER_MODEL_NAME = "openai/whisper-small"
DATASET_PATH = os.path.join(PROJECT_DIR, "data", "processed", f"asr_{SCRIPT_TARGET}")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "models", f"whisper-darija-{SCRIPT_TARGET}")
# Langue de tokenisation : Whisper ne connait pas "darija" nativement, on utilise
# "arabic" comme langue de base pour le tokenizer dans les deux cas (le darija partage
# l'essentiel de son vocabulaire/ecriture arabe avec le MSA pour le script arabe ; pour
# le script Latin, on utilise quand meme ce tokenizer de base car Whisper n'a pas de
# tokenizer "arabizi" dedie -- le fine-tuning va adapter le modele au nouveau vocabulaire).
WHISPER_LANGUAGE = "arabic"
WHISPER_TASK = "transcribe"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration : SCRIPT_TARGET={SCRIPT_TARGET}")
print(f"  Dataset source : {DATASET_PATH}")
print(f"  Sortie checkpoints : {OUTPUT_DIR}")

## 4. Chargement du dataset preprocesse

In [ ]:
dataset = load_from_disk(DATASET_PATH)
print(dataset)

# Verification rapide d'un exemple
print()
print("Exemple (texte cible):", dataset["train"][0]["sentence"])

## 5. Chargement du processor Whisper (feature extractor + tokenizer)

In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(WHISPER_MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(
    WHISPER_MODEL_NAME, language=WHISPER_LANGUAGE, task=WHISPER_TASK
)
processor = WhisperProcessor.from_pretrained(
    WHISPER_MODEL_NAME, language=WHISPER_LANGUAGE, task=WHISPER_TASK
)

print("Processor charge avec succes.")

## 6. Preparation des features (log-Mel spectrogram + tokenization)

Cette etape convertit chaque exemple audio en log-Mel spectrogram (entree du modele) et
chaque texte cible en sequence d'IDs de tokens (sortie attendue). C'est l'etape la plus
longue de tout le notebook pour de gros datasets -- compter plusieurs minutes.


In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # Extraction des features audio (log-Mel spectrogram), 16kHz attendu
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Tokenization du texte cible
    batch["labels"] = tokenizer(batch["sentence"]).input_ids

    return batch


dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1,  # mettre >1 si plusieurs CPU disponibles et pas de souci memoire
)

print("Preparation terminee.")
print(dataset)

## 7. Data collator (gestion du padding dynamique)

Whisper a besoin d'un collator personnalise car les inputs audio (spectrogrammes) et les
labels (texte) ont des strategies de padding differentes.


In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Partie audio : deja en log-Mel spectrogram, juste besoin de padding/conversion en tensor
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Partie texte : padding standard avec -100 pour ignorer les tokens de padding dans la loss
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        # Retrait du token de debut de sequence (BOS) s'il a ete ajoute par le collator
        # (le modele l'ajoute deja automatiquement pendant l'entrainement)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=None,  # sera defini juste apres le chargement du modele
)

## 8. Chargement du modele Whisper pre-entraine

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_NAME)

# On force la langue/tache au niveau de la config de generation (evite des warnings,
# et garantit que le modele genere bien dans la langue/tache voulue par defaut)
model.generation_config.language = WHISPER_LANGUAGE
model.generation_config.task = WHISPER_TASK
model.generation_config.forced_decoder_ids = None

# Mise a jour du decoder_start_token_id dans le data collator (defini une fois le modele charge)
data_collator.decoder_start_token_id = model.config.decoder_start_token_id

print("Modele charge :", WHISPER_MODEL_NAME)
print("Nombre de parametres :", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 9. Metrique d'evaluation (WER - Word Error Rate)

In [ ]:
wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Remplace -100 par le pad_token_id pour pouvoir decoder correctement
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

## 10. Configuration de l'entrainement

Hyperparametres adaptes pour GPU T4 (16GB VRAM) sur Colab gratuit. `save_steps` est volontairement
frequent pour permettre une reprise apres deconnexion sans perdre trop de progres.


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # batch effectif = 16
    learning_rate=1e-5,
    warmup_steps=200,
    num_train_epochs=3,
    gradient_checkpointing=True,        # reduit la VRAM utilisee, utile sur T4
    fp16=True,                          # entrainement en precision mixte, plus rapide sur T4
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=50,
    save_total_limit=3,                 # ne garde que les 3 derniers checkpoints (espace Drive)
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    push_to_hub=False,
)

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

## 11. Lancement de l'entrainement (avec reprise automatique si checkpoint existant)

Si la session a deja tourne partiellement avant une deconnexion, cette cellule detecte
le dernier checkpoint sauvegarde dans `OUTPUT_DIR` et reprend automatiquement depuis la.


In [ ]:
import glob

existing_checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))
resume_from = existing_checkpoints[-1] if existing_checkpoints else None

if resume_from:
    print(f"Reprise depuis le checkpoint : {resume_from}")
else:
    print("Aucun checkpoint existant, entrainement depuis le debut.")

trainer.train(resume_from_checkpoint=resume_from)

## 12. Sauvegarde du modele final + processor

In [ ]:
final_path = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_path)
processor.save_pretrained(final_path)

print(f"Modele final sauvegarde -> {final_path}")

## 13. Evaluation finale sur le test set

In [ ]:
test_results = trainer.evaluate(eval_dataset=dataset["test"], metric_key_prefix="test")
print(f"Resultats sur le test set ({SCRIPT_TARGET}) :")
for k, v in test_results.items():
    print(f"  {k}: {v}")

## 14. Exemples qualitatifs (predictions vs references)

In [ ]:
import torch

model.eval()
n_examples = 5

for i in range(n_examples):
    sample = dataset["test"][i]
    input_features = torch.tensor(sample["input_features"]).unsqueeze(0)
    if torch.cuda.is_available():
        input_features = input_features.to("cuda")
        model = model.to("cuda")

    with torch.no_grad():
        predicted_ids = model.generate(input_features, max_length=225)

    predicted_text = tokenizer.decode(predicted_ids[0], skip_special_tokens=True)

    label_ids = sample["labels"]
    label_ids = [tok if tok != -100 else tokenizer.pad_token_id for tok in label_ids]
    reference_text = tokenizer.decode(label_ids, skip_special_tokens=True)

    print(f"--- Exemple {i+1} ---")
    print(f"  Reference : {reference_text}")
    print(f"  Prediction: {predicted_text}")
    print()

## 15. Prochaines etapes

- Si `SCRIPT_TARGET = "latin"` vient de tourner : remonter a la cellule de configuration,
  changer `SCRIPT_TARGET = "arabic"`, et relancer depuis cette cellule jusqu'a la fin
  (Runtime > Redemarrer la session recommande pour repartir propre en VRAM).
- Une fois les DEUX modeles entraines : passer a `03_finetune_nllb_translation.ipynb`
- Noter le WER final de chaque modele pour le rapport.
